In [1]:
import requests
import pandas as pd

In [2]:
API_URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.cond": "Endometriosis",
    "pageSize": 5,
    "format": "json"
}

response = requests.get(API_URL, params=params, timeout=30)

print(response.status_code)

200


In [3]:
data = response.json()

data.keys()

dict_keys(['studies', 'nextPageToken'])

In [4]:
len(data["studies"])

5

In [5]:
data["studies"][0]

{'protocolSection': {'identificationModule': {'nctId': 'NCT06073379',
   'orgStudyIdInfo': {'id': 'POUHIN AOIparaM 2021'},
   'organization': {'fullName': 'Centre Hospitalier Universitaire Dijon',
    'class': 'OTHER'},
   'briefTitle': 'Efficacy of Korean Manupuncture on Pain in Women With Endometriosis: a Parallel-group Randomized Controlled Trial',
   'officialTitle': 'Efficacy of Korean Manupuncture on Pain in Women With Endometriosis: a Parallel-group Randomized Controlled Trial',
   'acronym': 'ENVOL'},
  'statusModule': {'statusVerifiedDate': '2026-02',
   'overallStatus': 'COMPLETED',
   'expandedAccessInfo': {'hasExpandedAccess': False},
   'startDateStruct': {'date': '2023-11-29', 'type': 'ACTUAL'},
   'primaryCompletionDateStruct': {'date': '2025-07-09', 'type': 'ACTUAL'},
   'completionDateStruct': {'date': '2025-07-09', 'type': 'ACTUAL'},
   'studyFirstSubmitDate': '2023-09-29',
   'studyFirstSubmitQcDate': '2023-10-05',
   'studyFirstPostDateStruct': {'date': '2023-10-10'

In [6]:
first_study = data["studies"][0]

protocol = first_study["protocolSection"]
protocol.keys()

dict_keys(['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule'])

In [7]:
identification = protocol["identificationModule"]

print("NCT ID:", identification["nctId"])
print("Title:", identification["briefTitle"])

NCT ID: NCT06073379
Title: Efficacy of Korean Manupuncture on Pain in Women With Endometriosis: a Parallel-group Randomized Controlled Trial


In [8]:
conditions = protocol.get("conditionsModule", {}).get("conditions", [])

print(conditions)

['Endometriosis', 'Pain']


In [9]:
def extract_basic_study_info(study: dict, searched_condition: str) -> dict:
    """Extract selected fields from one ClinicalTrials.gov study record."""

    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    design = protocol.get("designModule", {})
    conditions = protocol.get("conditionsModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    eligibility = protocol.get("eligibilityModule", {})

    enrollment_info = design.get("enrollmentInfo", {})
    lead_sponsor = sponsor.get("leadSponsor", {})

    return {
        "nct_id": identification.get("nctId"),
        "searched_condition": searched_condition,
        "brief_title": identification.get("briefTitle"),
        "listed_conditions": conditions.get("conditions", []),
        "study_type": design.get("studyType"),
        "overall_status": status.get("overallStatus"),
        "enrollment": enrollment_info.get("count"),
        "enrollment_type": enrollment_info.get("type"),
        "lead_sponsor": lead_sponsor.get("name"),
        "sponsor_class": lead_sponsor.get("class"),
        "sex": eligibility.get("sex"),
        "has_results": study.get("hasResults", False),
    }

In [10]:
records = [
    extract_basic_study_info(
        study=study,
        searched_condition="Endometriosis"
    )
    for study in data["studies"]
]

sample_df = pd.DataFrame(records)

sample_df

,nct_id,searched_condition,brief_title,listed_conditions,study_type,overall_status,enrollment,enrollment_type,lead_sponsor,sponsor_class,sex,has_results
0,NCT06073379,Endometriosis,Efficacy of Korean Manupuncture on Pain in Wom...,"[Endometriosis, Pain]",INTERVENTIONAL,COMPLETED,74,ACTUAL,Centre Hospitalier Universitaire Dijon,OTHER,ALL,False
1,NCT06168097,Endometriosis,The Use of MicroRNAs Dysregulation as Potentia...,[Endometriosis],INTERVENTIONAL,ACTIVE_NOT_RECRUITING,200,ESTIMATED,"Asian Institute of Gastroenterology, India",OTHER,ALL,False
2,NCT00761683,Endometriosis,Non-Interventional Study to Evaluate Effect of...,[Endometriosis],OBSERVATIONAL,TERMINATED,105,ESTIMATED,AstraZeneca,INDUSTRY,FEMALE,False
3,NCT04614246,Endometriosis,Study to Gather Information How Well Three Dif...,[Endometriosis],INTERVENTIONAL,TERMINATED,215,ACTUAL,Bayer,INDUSTRY,FEMALE,True
4,NCT03928288,Endometriosis,Cabergoline for the Treatment of Chronic Pain ...,[Endometriosis],INTERVENTIONAL,COMPLETED,129,ACTUAL,Boston Children's Hospital,OTHER,FEMALE,False


In [11]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   nct_id              5 non-null      object
 1   searched_condition  5 non-null      object
 2   brief_title         5 non-null      object
 3   listed_conditions   5 non-null      object
 4   study_type          5 non-null      object
 5   overall_status      5 non-null      object
 6   enrollment          5 non-null      int64 
 7   enrollment_type     5 non-null      object
 8   lead_sponsor        5 non-null      object
 9   sponsor_class       5 non-null      object
 10  sex                 5 non-null      object
 11  has_results         5 non-null      bool  
dtypes: bool(1), int64(1), object(10)
memory usage: 573.0+ bytes


In [12]:
sample_df[
    [
        "nct_id",
        "searched_condition",
        "study_type",
        "overall_status",
        "enrollment",
        "has_results"
    ]
]

,nct_id,searched_condition,study_type,overall_status,enrollment,has_results
0,NCT06073379,Endometriosis,INTERVENTIONAL,COMPLETED,74,False
1,NCT06168097,Endometriosis,INTERVENTIONAL,ACTIVE_NOT_RECRUITING,200,False
2,NCT00761683,Endometriosis,OBSERVATIONAL,TERMINATED,105,False
3,NCT04614246,Endometriosis,INTERVENTIONAL,TERMINATED,215,True
4,NCT03928288,Endometriosis,INTERVENTIONAL,COMPLETED,129,False


In [13]:
sample_df.to_csv(
    "../data/raw/endometriosis_sample.csv",
    index=False
)